# 02 - Baseline Experiments

This notebook establishes the baseline performance using TF-IDF + Logistic Regression.
The baseline sets the "floor" that SciBERT must beat by >5 AUROC points.

**Expected baseline AUROC: 0.65-0.72** (on full dataset)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from src.classifier import BaselineClassifier, train_baseline

## Load Data

In [ ]:
data_dir = Path('../data/processed')

train_df = pd.read_parquet(data_dir / 'train.parquet')
val_df = pd.read_parquet(data_dir / 'val.parquet')
test_df = pd.read_parquet(data_dir / 'test.parquet')

print(f'Train: {len(train_df)} samples')
print(f'Val: {len(val_df)} samples')
print(f'Test: {len(test_df)} samples')
print(f'\nTrain label distribution:\n{train_df["label"].value_counts()}')

## Explore Data Quality

In [ ]:
# Check text lengths
train_df['text_len'] = train_df['methods_text'].str.len()
print('Text length statistics:')
print(train_df['text_len'].describe())

In [ ]:
# Sample texts
for i in range(min(3, len(train_df))):
    row = train_df.iloc[i]
    print(f"\n--- {row['title']} (label={row['label']}) ---")
    print(row['methods_text'][:500])

## Train Baseline (TF-IDF + Logistic Regression)

In [ ]:
classifier = BaselineClassifier(max_features=5000, ngram_range=(1, 2))
train_metrics, val_metrics = classifier.train(train_df, val_df)

## Evaluate on Test Set

In [ ]:
test_metrics = classifier.evaluate(test_df)

print('\n' + '='*50)
print('BASELINE RESULTS SUMMARY')
print('='*50)
print(f'Test AUROC: {test_metrics["auroc"]:.4f}')
print(f'Test F1: {test_metrics["f1"]:.4f}')
print('='*50)

## Feature Analysis

In [ ]:
# Top features for each class
feature_names = classifier.vectorizer.get_feature_names_out()
coefficients = classifier.classifier.coef_[0]

# Sort by coefficient magnitude
sorted_idx = np.argsort(coefficients)

print('Top 20 features indicating NOT reproducible (negative coefficient):')
for idx in sorted_idx[:20]:
    print(f'  {feature_names[idx]}: {coefficients[idx]:.4f}')

print('\nTop 20 features indicating REPRODUCIBLE (positive coefficient):')
for idx in sorted_idx[-20:][::-1]:
    print(f'  {feature_names[idx]}: {coefficients[idx]:.4f}')

## Save Baseline Model

In [ ]:
classifier.save('../models/baseline')
print('Baseline model saved to models/baseline/')

## Conclusions

**Note:** With the current small sample dataset (30 papers), the model overfits.
Once the full Papers With Code dataset is available (~1200 papers), expect:

| Metric | Expected Baseline |
|--------|-------------------|
| AUROC | 0.65-0.72 |
| F1 (macro) | 0.60-0.65 |

SciBERT target: AUROC > 0.78 (must beat baseline by >5 points)